# Solving CartPole-v1 with PPO (from scratch)

This notebook demonstrates how to solve the CartPole-v1 environment using a custom implementation of the Proximal Policy Optimization (PPO) algorithm in PyTorch. No external RL libraries are used.

---

**Note:** All environment and hyperparameter settings are now collected in a single `CONFIG` dictionary at the top of the notebook. To change the environment or any hyperparameter, simply edit the values in the config cell.

## 1. Install and Import Required Libraries
We will use gymnasium and torch for this implementation.

In [1]:
%pip install gymnasium tsilva-notebook-utils

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 2. Set Up CartPole-v1 Environment
We will initialize the CartPole-v1 environment and display its basic information.

In [2]:
#ENV_ID = "CartPole-v1"
#ENV_ID = "Acrobot-v1"
#ENV_ID = "LunarLander-v3"
#ENV_ID = "Pendulum-v1"
ENV_ID = "MountainCar-v0"

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import numpy as np
import platform, os

# --- Config dictionary for all hyperparameters and environment settings ---
def setup_config(env_id):
    common = dict(
        env_id=env_id,         # Environment name
        seed=42,               # Random seed for reproducibility
        gamma=0.99,            # Discount factor for future rewards
        lam=0.95,              # GAE lambda for advantage estimation
        clip_epsilon=0.2,      # PPO clip range for policy update
        minibatch_size=64,     # Minibatch size for SGD
        episodes_per_epoch=20, # Number of episodes per training epoch
        eval_interval=5,       # Evaluate every N epochs
        eval_episodes=20,      # Number of episodes for evaluation
        reward_threshold=200,  # Reward threshold to consider environment solved
        policy_lr=3e-4,        # Learning rate for policy network
        value_lr=1e-3,         # Learning rate for value network
        hidden_dim=64,         # Hidden layer size for networks
        max_epochs=200,        # Maximum number of epochs to train
        entropy_coef=0.01,     # Coefficient for entropy bonus (encourages exploration)
    )
    env_specific = {
        "CartPole-v1": dict(
            gamma=0.99,           # Standard discount for CartPole
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for CartPole
            minibatch_size=32,    # Smaller batch for faster updates
            episodes_per_epoch=16,# Fewer episodes per epoch for quick feedback
            eval_interval=2,      # Evaluate more frequently for fast convergence
            eval_episodes=10,     # Fewer eval episodes for speed
            reward_threshold=475, # Official CartPole-v1 solved threshold
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=1e-3,        # Slightly higher for value net
            hidden_dim=64,        # Small net is sufficient for CartPole
            max_epochs=50,        # Should solve in fewer epochs
        ),
        "LunarLander-v3": dict(
            gamma=0.99,           # Standard discount for LunarLander
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for LunarLander
            minibatch_size=64,    # Larger batch for more stable updates
            episodes_per_epoch=8, # Fewer episodes per epoch (env is longer)
            eval_interval=2,      # Evaluate every 2 epochs
            eval_episodes=5,      # Fewer eval episodes for speed
            reward_threshold=200, # Solved threshold for LunarLander-v3
            policy_lr=1e-4,       # Lower LR for more complex env
            value_lr=5e-4,        # Lower LR for value net
            hidden_dim=128,       # Larger net for more complex env
            max_epochs=200,       # May require more epochs to solve
        ),
        "Acrobot-v1": dict(
            gamma=0.99,           # Standard discount for Acrobot
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for Acrobot
            minibatch_size=32,    # Smaller batch for faster updates
            episodes_per_epoch=16,# Fewer episodes per epoch for quick feedback
            eval_interval=2,      # Evaluate more frequently for fast convergence
            eval_episodes=10,     # Fewer eval episodes for speed
            reward_threshold=-100,# Solved threshold for Acrobot-v1 (average reward > -100)
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=1e-3,        # Slightly higher for value net
            hidden_dim=64,        # Small net is sufficient for Acrobot
            max_epochs=100,       # Should solve in fewer epochs
        ),
        "Pendulum-v1": dict(
            gamma=0.99,           # Standard discount for Pendulum
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for Pendulum
            minibatch_size=64,    # Larger batch for continuous action
            episodes_per_epoch=8, # Fewer episodes per epoch (env is longer)
            eval_interval=2,      # Evaluate every 2 epochs
            eval_episodes=5,      # Fewer eval episodes for speed
            reward_threshold=-200,# Solved threshold for Pendulum-v1 (average reward > -200)
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=1e-3,        # Slightly higher for value net
            hidden_dim=128,       # Larger net for continuous control
            max_epochs=200,       # May require more epochs to solve
        ),
        "MountainCar-v0": dict(
            gamma=0.99,           # Standard discount for MountainCar
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for MountainCar
            minibatch_size=32,    # Smaller batch for faster updates
            episodes_per_epoch=16,# Fewer episodes per epoch for quick feedback
            eval_interval=2,      # Evaluate more frequently for fast convergence
            eval_episodes=10,     # Fewer eval episodes for speed
            reward_threshold=-110,# Solved threshold for MountainCar-v0 (average reward > -110)
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=1e-3,        # Slightly higher for value net
            hidden_dim=64,        # Small net is sufficient for MountainCar
            max_epochs=100,       # Should solve in fewer epochs
            entropy_coef=0.00     # Encourage exploration for sparse rewards
        ),
    }
    if env_id not in env_specific:
        raise ValueError(f"Unsupported env_id: {env_id}")
    return {**common, **env_specific[env_id]}

CONFIG = setup_config(ENV_ID)

# --- Runtime metadata (optional, for reproducibility/info) ---
def runtime_metadata():
    return {
        "python_version": platform.python_version(),
        "run_host": os.uname().nodename,
    }
CONFIG.update(runtime_metadata())

# Set up the environment
env = gym.make(CONFIG['env_id'])
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.n
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

Observation space: Box([-1.2  -0.07], [0.6  0.07], (2,), float32)
Action space: Discrete(3)


## 3. Implement PPO Agent
We will define the policy and value networks, and the PPO update step.

In [4]:
# Policy and Value Networks
class PolicyNet(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_dim=None):
        super().__init__()
        h = hidden_dim or CONFIG['hidden_dim']
        self.net = nn.Sequential(
            nn.Linear(obs_dim, h), nn.Tanh(),
            nn.Linear(h, h), nn.Tanh(),
            nn.Linear(h, act_dim)
        )
    def forward(self, x):
        return self.net(x)

class ValueNet(nn.Module):
    def __init__(self, obs_dim, hidden_dim=None):
        super().__init__()
        h = hidden_dim or CONFIG['hidden_dim']
        self.net = nn.Sequential(
            nn.Linear(obs_dim, h), nn.Tanh(),
            nn.Linear(h, h), nn.Tanh(),
            nn.Linear(h, 1)
        )
    def forward(self, x):
        return self.net(x)

## 4. Train PPO Agent
We will train the PPO agent on CartPole-v1.

In [5]:
import torch
import gymnasium as gym
import numpy as np
from torch.distributions.categorical import Categorical


def collect_rollouts(policy, env, n_episodes=1, value_net=None, deterministic=False, render=False, seed=None):
    """
    Collect rollouts using the given policy and environment.
    Returns: list of episode rewards, and (obs, actions, rewards, dones, logps, values) for all steps.
    """
    from torch.distributions.categorical import Categorical
    episode_rewards = []
    obs_buf, act_buf, rew_buf, done_buf, logp_buf, val_buf = [], [], [], [], [], []
    if seed is not None:
        obs, info = env.reset(seed=seed)
    else:
        obs, info = env.reset()
    episodes_collected = 0
    ep_rews = []
    while episodes_collected < n_episodes:
        obs_t = torch.as_tensor(obs, dtype=torch.float32)
        logits = policy(obs_t)
        dist = Categorical(logits=logits)
        if deterministic:
            action = dist.probs.argmax().item()
            logp = dist.log_prob(torch.tensor(action))
        else:
            action = dist.sample().item()
            logp = dist.log_prob(torch.tensor(action))
        value_pred = value_net(obs_t).item() if value_net is not None else 0
        obs_buf.append(obs)
        act_buf.append(action)
        logp_buf.append(logp.item())
        val_buf.append(value_pred)
        next_obs, reward, terminated, truncated, info = env.step(action)
        done_flag = terminated or truncated
        rew_buf.append(reward)
        done_buf.append(done_flag)
        ep_rews.append(reward)
        if render:
            env.render()
        obs = next_obs
        if done_flag:
            episode_rewards.append(sum(ep_rews))
            obs, info = env.reset()
            ep_rews = []
            episodes_collected += 1
    return episode_rewards, (obs_buf, act_buf, rew_buf, done_buf, logp_buf, val_buf)

# TODO: generalize into method that can also render N episodes
def evaluate_policy(policy, env_name=None, n_episodes=None):
    env_name = env_name or CONFIG['env_id']
    n_episodes = n_episodes or CONFIG['eval_episodes']
    eval_env = gym.make(env_name)
    eval_seed = np.random.randint(0, 1000000)
    episode_rewards, _ = collect_rollouts(policy, eval_env, n_episodes=n_episodes, deterministic=True, render=False, seed=eval_seed)
    mean_reward = np.mean(episode_rewards)
    return mean_reward, episode_rewards

evaluate_policy(PolicyNet(obs_dim, act_dim), n_episodes=5)

(np.float64(-200.0), [-200.0, -200.0, -200.0, -200.0, -200.0])

In [ ]:
# Helper functions for PPO
import random
import torch.optim as optim

def set_global_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    advantages = []
    gae = 0
    values = values + [0]
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + gamma * values[t+1] * (1 - dones[t]) - values[t]
        gae = delta + gamma * lam * (1 - dones[t]) * gae
        advantages.insert(0, gae)
    return advantages

# Set fixed seed for training
set_global_seed(CONFIG['seed'])
obs, info = env.reset(seed=CONFIG['seed'])

# PPO Training Loop
entropy_coef = CONFIG.get('entropy_coef', 0.0)
policy_net = PolicyNet(obs_dim, act_dim, hidden_dim=CONFIG['hidden_dim'])
value_net = ValueNet(obs_dim, hidden_dim=CONFIG['hidden_dim'])
policy_optim = optim.Adam(policy_net.parameters(), lr=CONFIG['policy_lr'])
value_optim = optim.Adam(value_net.parameters(), lr=CONFIG['value_lr'])

clip_epsilon = CONFIG['clip_epsilon']
minibatch_size = CONFIG['minibatch_size']
gamma = CONFIG['gamma']
lam = CONFIG['lam']
episodes_per_epoch = CONFIG['episodes_per_epoch']
eval_interval = CONFIG['eval_interval']
eval_episodes = CONFIG['eval_episodes']
reward_threshold = CONFIG['reward_threshold']
#max_epochs = 200  # Safety cap

train_rewards = []  # Store mean reward per epoch
solved = False
current_epoch = 0

while not solved:# and current_epoch < max_epochs:
    # Use collect_rollouts for training
    
    episode_rewards, (obs_buf, act_buf, rew_buf, done_buf, logp_buf, val_buf) = collect_rollouts(
        policy_net, env, n_episodes=episodes_per_epoch, value_net=value_net, deterministic=False, render=False, seed=CONFIG['seed'])
    train_rewards.append(np.mean(episode_rewards))
    adv_buf = compute_gae(rew_buf, val_buf, done_buf, gamma, lam)
    ret_buf = (np.array(adv_buf) + np.array(val_buf)).tolist()
    # Convert buffers to tensors
    obs_tensor = torch.as_tensor(np.array(obs_buf), dtype=torch.float32)
    act_tensor = torch.as_tensor(np.array(act_buf), dtype=torch.int64)
    adv_tensor = torch.as_tensor(np.array(adv_buf), dtype=torch.float32)
    ret_tensor = torch.as_tensor(np.array(ret_buf), dtype=torch.float32)
    logp_old_tensor = torch.as_tensor(np.array(logp_buf), dtype=torch.float32)
    # Normalize advantages
    adv_tensor = (adv_tensor - adv_tensor.mean()) / (adv_tensor.std() + 1e-8)
    # PPO update
    num_samples = len(obs_buf)
    for _ in range(10):
        idx = np.random.permutation(num_samples)
        for start in range(0, num_samples, minibatch_size):
            end = start + minibatch_size
            mb_idx = idx[start:end]
            logits = policy_net(obs_tensor[mb_idx])
            dist = Categorical(logits=logits)
            logp = dist.log_prob(act_tensor[mb_idx])
            ratio = torch.exp(logp - logp_old_tensor[mb_idx])
            surr1 = ratio * adv_tensor[mb_idx]
            surr2 = torch.clamp(ratio, 1 - clip_epsilon, 1 + clip_epsilon) * adv_tensor[mb_idx]
            entropy = dist.entropy().mean()
            policy_loss = -torch.min(surr1, surr2).mean() - entropy_coef * entropy
            value_pred = value_net(obs_tensor[mb_idx]).squeeze()
            value_loss = ((ret_tensor[mb_idx] - value_pred) ** 2).mean()
            policy_optim.zero_grad()
            policy_loss.backward()
            policy_optim.step()
            value_optim.zero_grad()
            value_loss.backward()
            value_optim.step()
    current_epoch += 1
    print(f"Epoch {current_epoch} complete. Mean train reward: {train_rewards[-1]:.2f}")
    if current_epoch % eval_interval == 0:
        mean_eval_reward, _ = evaluate_policy(policy_net, n_episodes=eval_episodes)
        print(f"Evaluation after {current_epoch} epochs: Mean reward = {mean_eval_reward:.2f}")
        if mean_eval_reward >= reward_threshold:
            print(f"Solved! Mean evaluation reward {mean_eval_reward:.2f} >= {reward_threshold}")
            solved = True


Epoch 1 complete. Mean train reward: -200.00
Epoch 2 complete. Mean train reward: -200.00
Epoch 2 complete. Mean train reward: -200.00
Evaluation after 2 epochs: Mean reward = -200.00
Evaluation after 2 epochs: Mean reward = -200.00
Epoch 3 complete. Mean train reward: -200.00
Epoch 3 complete. Mean train reward: -200.00
Epoch 4 complete. Mean train reward: -200.00
Epoch 4 complete. Mean train reward: -200.00
Evaluation after 4 epochs: Mean reward = -200.00
Evaluation after 4 epochs: Mean reward = -200.00
Epoch 5 complete. Mean train reward: -200.00
Epoch 5 complete. Mean train reward: -200.00
Epoch 6 complete. Mean train reward: -200.00
Epoch 6 complete. Mean train reward: -200.00
Evaluation after 6 epochs: Mean reward = -200.00
Evaluation after 6 epochs: Mean reward = -200.00
Epoch 7 complete. Mean train reward: -200.00
Epoch 7 complete. Mean train reward: -200.00
Epoch 8 complete. Mean train reward: -200.00
Epoch 8 complete. Mean train reward: -200.00
Evaluation after 8 epochs: Mean

In [ ]:
# Plot training rewards after training
import matplotlib.pyplot as plt
plt.plot(train_rewards, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Mean Episode Reward')
plt.title('Training Reward per Epoch')
plt.grid(True)
plt.show()

In [ ]:
evaluate_policy(policy_net, n_episodes=100)

In [ ]:
from tsilva_notebook_utils.gymnasium import render_episode
from tsilva_notebook_utils.gymnasium import build_env

build_env_fn = lambda **kwargs: build_env(CONFIG['env_id'], **kwargs)

render_episode(
    env=build_env_fn,
    model=policy_net
)